Encapsulación del ColumnTransformer en src/preprocessing.py

In [4]:
%%writefile ../src/preprocessing.py
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer


def build_preprocessor(num_cols, cat_cols):

    num_pipe = Pipeline(steps=[
        ("imp", SimpleImputer(strategy="median")),
        ("sc", StandardScaler())
    ])

    cat_pipe = Pipeline(steps=[
        ("imp", SimpleImputer(strategy="most_frequent")),
        ("oh", OneHotEncoder(handle_unknown="ignore"))
    ])

    preproc = ColumnTransformer(transformers=[
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols)
    ])

    return preproc

Overwriting ../src/preprocessing.py


Verificamos que exista la carpeta src

In [5]:
import os
print(os.listdir("../src"))

['preprocessing.py', '__pycache__']




Aplicación de ColumnTransformer y validación del pipeline con fit/transform

In [6]:
import pandas as pd

df = pd.read_csv('../data/processed/04_default_credit_balanced_RandomOverSampler.csv')

X = df.drop(columns=["default payment next month", "ID"])
y = df["default payment next month"]

import sys
sys.path.append("..")

from src.preprocessing import build_preprocessor

# detectar columnas
num_cols = X.select_dtypes(include=["int64", "float64", "bool"]).columns
cat_cols = X.select_dtypes(include=["object", "category"]).columns

# usar función encapsulada
preproc = build_preprocessor(num_cols, cat_cols)

preproc.fit(X)

X_transformed = preproc.transform(X)

df_final = pd.DataFrame(X_transformed)
df_final.to_csv("../data/processed/dataset_pipeline.csv", index=False)

print("Dataset guardado y transformación exitosa:", X_transformed.shape)

Dataset guardado y transformación exitosa: (37382, 48)


Persistencia del pipeline de preprocesamiento con joblib

In [7]:
import joblib

joblib.dump(preproc, "../models/preprocessor.pkl")

['../models/preprocessor.pkl']